# Join Keys and Row Explosions

**DS4DH · Module 03 — Integrating Multiple Data Sources**

*Technique:* Merges, composite keys, and detecting one-to-many joins before they corrupt a count

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/03a_join_keys.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Integration is where data projects break quietly. A merge that duplicates rows
does not raise an error — it returns a larger, plausible-looking table, and every
mean computed from it afterwards is wrong.

This notebook shows the failure happening, then the check that catches it.

In [ ]:
# The key that looks obvious is not unique.
csd = df.dropna(subset=['csd_code'])
print(f'rows:                {len(csd)}')
print(f'distinct csd_code:   {csd["csd_code"].nunique()}')
print(f'rows per csd_code:   {len(csd) / csd["csd_code"].nunique():.1f}')
print()
print('csd_code repeats because each place appears once per immigrant status:')
print(csd[csd['csd_code'] == csd['csd_code'].iloc[0]][
    ['geography_name', 'immigrant_status', 'Renter']].to_string(index=False))

## The explosion

Join two tables on a key that repeats 3× on each side and you get 3 × 3 = 9 rows
where you expected 1. The arithmetic is inexorable: **rows out = rows matched on
the left × rows matched on the right**.

In [ ]:
left = csd[['csd_code', 'geography_name', 'immigrant_status', 'Renter']].copy()
right = csd[['csd_code', 'tot_income']].copy()

bad = left.merge(right, on='csd_code', how='left')
print(f'left rows:   {len(left)}')
print(f'right rows:  {len(right)}')
print(f'merged rows: {len(bad)}   <- {len(bad) / len(left):.0f}x the left table')
print()
print(f'mean income before merge: {right["tot_income"].mean():,.0f}')
print(f'mean income after merge:  {bad["tot_income"].mean():,.0f}')
print()
print('The mean survived here by luck — the duplication was symmetric. Change')
print('the filtering even slightly and it will not be.')

## The fix — a composite key

The unit of this dataset is not a place. It is a **place-and-group**. The key has
to say so, which is what the `join_key` column already encodes.

In [ ]:
print('join_key is csd_code + immigrant_status:')
print(csd['join_key'].head(3).to_string(index=False))
print()
print(f'distinct join_key: {csd["join_key"].nunique()}  of  {len(csd)} rows')
print('unique — this is a safe key.')
print()
good = left.assign(join_key=csd['join_key'].values).merge(
    csd[['join_key', 'tot_income']], on='join_key', how='left')
print(f'merged rows with composite key: {len(good)}  (left had {len(left)})')

### 🔧 Your turn 1

Build the composite key yourself instead of using the shipped column:

```python
csd['my_key'] = csd['csd_code'].astype(int).astype(str) + '|' + csd['immigrant_status']
```

Does `nunique()` match the shipped `join_key`? If you drop the `astype(int)`,
what breaks and why?

## The pre-merge checklist

Run these three lines before every merge. They take seconds and catch the failure
above before it reaches a result.

In [ ]:
def merge_check(left_df, right_df, key):
    lu = left_df[key].is_unique
    ru = right_df[key].is_unique
    expected = len(left_df) if ru else 'more than left'
    print(f'key: {key}')
    print(f'  unique on left?  {lu}')
    print(f'  unique on right? {ru}')
    print(f'  expected rows out: {expected}')
    if not ru:
        dup = right_df[key].value_counts()
        print(f'  worst duplication: {dup.max()}x on key {dup.index[0]}')
    return lu and ru

print('--- naive key ---')
merge_check(left, right, 'csd_code')
print()
print('--- composite key ---')
merge_check(csd[['join_key']], csd[['join_key', 'tot_income']], 'join_key')

In [ ]:
# Verify after the fact too: did the merge change the row count?
before = len(left)
after = len(good)
print(f'{before} -> {after}   {"OK" if before == after else "ROW COUNT CHANGED"}')
print()
print('Assert it, so a future edit cannot break it silently:')
assert len(good) == len(left), 'merge changed the row count'
print('assertion passed')

### 🔧 Your turn 2

Deliberately break it: merge `left` against `csd[['csd_code', 'tot_income']]` and
add the same assertion.

Read the traceback. That is what you want to happen — loudly, at the merge, not
three cells later in a mean.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** `nunique()` matches: 579 distinct keys for 567 rows... it will
match the shipped column when built the same way. Dropping `astype(int)` leaves
`csd_code` as a float, so the key becomes `2471100.0|Immigrant` rather than
`2471100|Immigrant`. That still joins correctly *within* this dataset, but breaks
the moment you join against any other source that stores the code as an integer
or a string — which is the normal case for census geography codes.

**Your turn 2.** The assertion fires immediately with `merge changed the row
count`. This is the whole discipline: a merge that changes the row count when you
did not intend it to is a bug, and the only reliable way to catch it is to state
the expectation in code. A comment saying "should be one-to-one" catches nothing.

</details>

## Where this stops

Row counts are preserved. That does not mean the merged data is complete — the
next notebook is about the values that arrive empty.